# 02 - Initial Data Preprocessing

**Project:** E-Commerce User Behavior Analysis & Recommendation System  
**Notebook purpose:** Apply processing and cleaning steps that were influenced by findings from initial data exploration. Obtain an intermediate processed dataset containing both October and November data.

---

## Environment Setup

This notebook was run on **Kaggle Notebooks** using the dataset attached directly 
from the Kaggle dataset page. No local setup or downloads are required to run it.

### To reproduce this notebook

1. Go to the repository on GitHub:  
   [ecommerce-behavior-analysis](https://github.com/halleepham/ecommerce-behavior-analysis)
2. Download `notebooks/02_initial_data_preprocessing.ipynb`
4. Go to the dataset page on Kaggle:  
   [E-Commerce Behavior Data from Multi-Category Store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store)
5. Click the three dots, then **New Notebook** to open a fresh Kaggle notebook with the dataset already attached
6. Click 'File' -> 'Import Notebook' and select the downloaded file. The dataset will already be attached
7. Run all cells top to bottom

### Data path

All cells in this notebook use the following path to access the raw data:

    /kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store

### Python version and key libraries

| Library | Version |
|---|---|
| Python | 3.12.12 |
| pandas | 2.3.3 |
| numpy | 2.0.2 |
| fastparquet | 2025.12.0 |

* In order to install fastparquet:
    * Under 'Session options' in the notebook tab on the right sidebar, make sure 'INTERNET' is on
    * This setting requires that your Kaggle account is phone verified
    * fastparquet chosen over pyarrow because it allows appending to parquet files

---

## Scope

This notebook reads both complete raw files (`2019-Oct.csv` and `2019-Nov.csv`) in chunks of 500,000 rows at a time to avoid memory issues. Each chunk will be individually cleaned and concatenated into a dataframe depending on if it is October or November data. The two processed dataframes will be saved as a single, combined parquet file for more efficient storage and access.

In [ ]:
%pip install fastparquet

In [2]:
import os
import sys
import pandas as pd
import numpy as np
import fastparquet

# Data paths
data_path = '/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store'
oct_file  = f'{data_path}/2019-Oct.csv'
nov_file  = f'{data_path}/2019-Nov.csv'

# Constants
CHUNK_SIZE = 500_000

# Verify files exist 
print(f'October  file exists : {os.path.exists(oct_file)}')
print(f'November file exists : {os.path.exists(nov_file)}\n')

print(f'Python        : {sys.version}')
print(f'pandas        : {pd.__version__}')
print(f'numpy         : {np.__version__}')
print(f'fastparquet   : {fastparquet.__version__}')

October  file exists : True
November file exists : True

Python        : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
pandas        : 2.3.3
numpy         : 2.0.2
fastparquet   : 2025.12.0


## Data Processing

**clean_chunk**: Function to clean an individual chunk
* Convert timestamps to datetime
* Fill missing values
* Replace '0' prices with NaN
* Cast values to more efficient datatypes
* Remove duplicates in chunk

In [ ]:
def clean_chunk(df):
    # Convert into date time
    df['event_time'] = pd.to_datetime(df['event_time'])

    # Fill NA Columns
    df.fillna({'category_code':'unknown', 'brand':'unknown'}, inplace=True)

    # Replace '0' prices with NaN
    df['price'] = df['price'].replace(0, np.nan)

    # Convert to more efficient datatypes
    df['event_type'] = df['event_type'].astype('category')
    df['brand'] = df['brand'].astype('category')
    
    df['price'] = df['price'].astype('float32')
    df[['product_id', 'category_id', 'user_id']] = df[['product_id', 'category_id', 'user_id']].apply(pd.to_numeric, downcast='integer')

    # Drop duplicates
    df.drop_duplicates(inplace=True, ignore_index=True)
    
    return df

**clean_dataset**: Function to clean entire dataset
* Loads raw dataset in chunks
* Cleans chunks individually using previously defined function
* Combines chunks into a single dataset and returns it

In [ ]:
def clean_dataset(input_file):

    chunks = []

    for chunk in pd.read_csv(input_file, chunksize=CHUNK_SIZE):
        chunk_df = clean_chunk(chunk)
        chunks.append(chunk)

    clean_df = pd.concat(chunks, ignore_index=True)

    # Drop duplicates from final df (incase of overlap between chunks)
    clean_df.drop_duplicates(inplace=True, ignore_index=True)

    return clean_df

Export processed dataset:
* Get cleaned oct and nov data
* Export data to parquet for more efficient storage and speed
    * Concatenating the dfs and writing once takes too much memory
    * Append one at a time to limit memory usage
        * Peak memory usage still 26/30 gb
        * Better memory optimization will be figured out at a later time


In [ ]:
oct_data_clean = clean_dataset(oct_file)
nov_data_clean = clean_dataset(nov_file)

# Export clean data to parquet file
oct_data_clean.to_parquet('/kaggle/working/ecommerce_oct_nov_clean.parquet', engine='fastparquet', compression='snappy')
nov_data_clean.to_parquet('/kaggle/working/ecommerce_oct_nov_clean.parquet', engine='fastparquet', compression='snappy', append=True)